# Regex 101 — การเขียน Regular Expression

**Regex (Regular Expression)** คือภาษาเล็กๆ ที่ใช้สำหรับ "ค้นหา" หรือ "จับคู่" รูปแบบของข้อความ (pattern matching)

ในงาน Cybersecurity / Splunk เราใช้ regex บ่อยมาก เช่น
- ดึง IP address ออกจาก log
- หา email ที่อยู่ในไฟล์ data dump
- แยก field ออกจาก raw log (ใช้ `rex` command ใน Splunk)
- ตรวจ pattern ของ attack เช่น SQL injection, XSS

เครื่องมือฝึก (แนะนำให้เปิดควบคู่):
- [regex101.com](https://regex101.com) — ทดสอบ regex แบบ real-time พร้อมคำอธิบาย
- [regexr.com](https://regexr.com) — สำหรับฝึกอีกตัว

---

## สารบัญ

1. Literal characters — ตัวอักษรตรงๆ
2. Metacharacters — ตัวอักษรพิเศษ
3. Character classes — กลุ่มของตัวอักษร
4. Quantifiers — จำนวนครั้งที่ซ้ำ
5. Anchors — ตำแหน่งในข้อความ
6. Groups & Alternation — การจัดกลุ่มและทางเลือก
7. Lookahead / Lookbehind
8. ตัวอย่างจริงใน Splunk
9. แบบฝึกหัด

## 1. Literal Characters — ตัวอักษรตรงๆ

regex ที่ง่ายที่สุด คือพิมพ์ตัวอักษรไปตรงๆ จะ match กับข้อความที่เหมือนกัน

| Pattern | Match | ไม่ Match |
|---------|-------|-----------|
| `cat`   | "cat", "category", "concatenate" | "dog", "Cat" (case-sensitive) |
| `123`   | "123abc", "x123" | "12", "1 2 3" |

> **หมายเหตุ:** โดย default regex จะ case-sensitive — ถ้าอยากไม่แคร์ตัวพิมพ์ใหญ่/เล็ก ต้องใช้ flag `i` (เช่น `/cat/i` หรือใน Python ใช้ `re.IGNORECASE`)

In [ ]:
import re

text = "The cat sat on the mat. Cats are cute."

# match แบบ case-sensitive
print(re.findall(r"cat", text))   # ['cat', 'cat']  (จับ "cat" และจาก "Cats")

# ลอง case-insensitive
print(re.findall(r"cat", text, re.IGNORECASE))   # ['cat', 'Cat']

## 2. Metacharacters — ตัวอักษรพิเศษ

ตัวเหล่านี้ "ไม่ใช่" ตัวอักษรธรรมดา แต่มีความหมายพิเศษใน regex

| Char | ความหมาย | ตัวอย่าง |
|------|---------|---------|
| `.`  | match ตัวอักษรอะไรก็ได้ 1 ตัว (ยกเว้น newline) | `c.t` → "cat", "cut", "c@t" |
| `\d` | digit (0-9) | `\d\d` → "42", "99" |
| `\D` | non-digit (ตรงข้าม \d) | `\D` → "a", "!", " " |
| `\w` | word char (a-z, A-Z, 0-9, _) | `\w+` → "hello_world" |
| `\W` | non-word char | `\W` → " ", "!", "@" |
| `\s` | whitespace (space, tab, newline) | `\s+` → "   " |
| `\S` | non-whitespace | `\S+` → "abc123" |
| `\b` | word boundary (รอยต่อระหว่าง word/non-word) | `\bcat\b` → "cat" แต่ไม่ใช่ "category" |

### Escape — ถ้าอยาก match ตัวพิเศษตรงๆ

ใช้ `\` นำหน้า เช่น
- `\.` → จุดจริงๆ
- `\?` → เครื่องหมายคำถามจริงๆ
- `\\` → backslash

> **กฎจำง่าย:** ตัวอักษรเล็ก (`\d \w \s`) = "ใช่", ตัวอักษรใหญ่ (`\D \W \S`) = "ไม่ใช่"

In [ ]:
import re

# . match ตัวอักษรอะไรก็ได้ 1 ตัว
print(re.findall(r"c.t", "cat cut c@t cot c t"))   # ['cat', 'cut', 'c@t', 'cot']
# สังเกต "c t" ไม่ติด เพราะ . ไม่ match newline แต่ space match ได้นะ
# จริงๆ "c t" ติดด้วย เพราะ ' ' คือตัวอักษร 1 ตัว
print(re.findall(r"c.t", "c t"))   # ['c t']

# \d match digit
print(re.findall(r"\d+", "Order #1234 costs $56.78"))   # ['1234', '56', '78']

# \w match word char
print(re.findall(r"\w+", "hello, world! foo_bar 123"))   # ['hello', 'world', 'foo_bar', '123']

# escape — จุดจริงๆ
print(re.findall(r"\d+\.\d+", "version 1.2 and 3.14"))   # ['1.2', '3.14']

# \b word boundary
print(re.findall(r"\bcat\b", "cat category concat scatter"))   # ['cat']

## 3. Character Classes — กลุ่มตัวอักษร `[ ]`

ใช้ `[...]` เพื่อบอกว่า "match ตัวใดตัวหนึ่งในนี้"

| Pattern | ความหมาย |
|---------|---------|
| `[abc]` | match a หรือ b หรือ c (ตัวเดียว) |
| `[a-z]` | a ถึง z (ตัวพิมพ์เล็ก) |
| `[A-Z]` | A ถึง Z (ตัวพิมพ์ใหญ่) |
| `[0-9]` | 0 ถึง 9 (เทียบเท่า `\d`) |
| `[a-zA-Z0-9]` | ตัวอักษรหรือตัวเลข |
| `[^abc]` | **ไม่ใช่** a, b, c (`^` ใน `[]` = ปฏิเสธ) |
| `[.?!]` | match จุด, ?, ! (ใน `[]` ไม่ต้อง escape ตัวพิเศษส่วนใหญ่) |

### ตัวอย่าง
- `[Cc]at` → "Cat" หรือ "cat"
- `[aeiou]` → vowel ตัวใดตัวหนึ่ง
- `[^0-9]` → อะไรก็ได้ที่ไม่ใช่ตัวเลข

In [ ]:
import re

# match Cat หรือ cat
print(re.findall(r"[Cc]at", "Cat and cat are CATs"))   # ['Cat', 'cat']

# vowel
print(re.findall(r"[aeiou]", "Hello World"))   # ['e', 'o', 'o']

# ไม่ใช่ตัวเลข
print(re.findall(r"[^0-9]+", "abc123def456"))   # ['abc', 'def']

# hex color (#FFAA00, #1a2b3c)
print(re.findall(r"#[0-9a-fA-F]{6}", "Colors: #FF0000, #00ff00, #abcDEF"))
# ['#FF0000', '#00ff00', '#abcDEF']

## 4. Quantifiers — บอกจำนวนครั้งที่ซ้ำ

ใส่ "หลัง" ตัวอักษร/กลุ่ม เพื่อบอกว่าจะให้ซ้ำกี่ครั้ง

| Quantifier | ความหมาย | ตัวอย่าง |
|------------|---------|---------|
| `*`     | 0 ครั้งหรือมากกว่า | `a*` → "", "a", "aaa" |
| `+`     | 1 ครั้งหรือมากกว่า | `a+` → "a", "aaa" (ไม่ match "") |
| `?`     | 0 หรือ 1 ครั้ง (มีก็ได้ ไม่มีก็ได้) | `colou?r` → "color", "colour" |
| `{n}`   | ซ้ำเป๊ะๆ n ครั้ง | `\d{4}` → "2024" |
| `{n,}`  | อย่างน้อย n ครั้ง | `\d{3,}` → "123", "1234567" |
| `{n,m}` | n ถึง m ครั้ง | `\d{2,4}` → "12", "123", "1234" |

### Greedy vs Lazy

โดย default quantifiers จะ **greedy** = พยายาม match ให้ "ยาวที่สุด"

ถ้าเติม `?` ต่อท้าย จะกลายเป็น **lazy** = match "สั้นที่สุด"

| Pattern | Input | Match |
|---------|-------|-------|
| `<.+>`  (greedy) | `<b>hi</b>` | `<b>hi</b>` (ทั้งก้อน!) |
| `<.+?>` (lazy)   | `<b>hi</b>` | `<b>` แล้วก็ `</b>` |

> **เคล็ดลับ:** เวลาดึงข้อมูลใน HTML/log ที่มี delimiter ซ้ำๆ ให้ใช้ lazy (`.+?` หรือ `.*?`) เกือบทุกครั้ง

In [ ]:
import re

# * vs +
print(re.findall(r"ab*", "a ab abb abbb"))     # ['a', 'ab', 'abb', 'abbb']  (a อย่างเดียวก็ติด)
print(re.findall(r"ab+", "a ab abb abbb"))     # ['ab', 'abb', 'abbb']        (ต้องมี b อย่างน้อย 1)

# ? = optional
print(re.findall(r"colou?r", "color and colour"))   # ['color', 'colour']

# {n,m}
print(re.findall(r"\d{2,4}", "1 22 333 4444 55555"))   # ['22', '333', '4444', '5555']
# สังเกต 55555 ถูกตัดเป็น 5555 (greedy แต่จำกัด max 4)

# Greedy vs Lazy — ตัวอย่างคลาสสิก
html = "<b>bold</b> and <i>italic</i>"
print(re.findall(r"<.+>", html))    # ['<b>bold</b> and <i>italic</i>']   ← Greedy ดูดทั้งหมด!
print(re.findall(r"<.+?>", html))   # ['<b>', '</b>', '<i>', '</i>']      ← Lazy ดีกว่า

## 5. Anchors — ระบุ "ตำแหน่ง"

Anchors ไม่ match ตัวอักษร แต่ match "ตำแหน่ง"

| Anchor | ความหมาย |
|--------|---------|
| `^`    | จุดเริ่มของบรรทัด/string |
| `$`    | จุดท้ายของบรรทัด/string |
| `\b`   | word boundary |
| `\B`   | ไม่ใช่ word boundary |

### ตัวอย่าง

- `^Error` → match "Error" ที่ขึ้นต้นบรรทัดเท่านั้น
- `\.log$` → match ที่ลงท้ายด้วย ".log"
- `^\d+$` → match string ที่เป็นตัวเลขล้วนๆ ทั้งบรรทัด

> **ระวัง:** `^` มี 2 ความหมาย
> - นอก `[]` = anchor (ต้นบรรทัด)
> - ใน `[]` = ปฏิเสธ เช่น `[^abc]`

In [ ]:
import re

logs = """ERROR: failed login
INFO: user logged in
ERROR: timeout
WARN: high memory"""

# match บรรทัดที่ขึ้นต้นด้วย ERROR (ต้องใช้ flag MULTILINE)
print(re.findall(r"^ERROR.*", logs, re.MULTILINE))
# ['ERROR: failed login', 'ERROR: timeout']

# string เป็นตัวเลขล้วนๆ
def is_all_digits(s):
    return bool(re.match(r"^\d+$", s))

print(is_all_digits("12345"))   # True
print(is_all_digits("12a45"))   # False
print(is_all_digits(""))        # False (เพราะ + ต้องมีอย่างน้อย 1)

# ลงท้ายด้วย .log
files = ["app.log", "error.txt", "system.log", "data.log.bak"]
print([f for f in files if re.search(r"\.log$", f)])   # ['app.log', 'system.log']

## 6. Groups `( )` และ Alternation `|`

### Groups `( ... )` — จัดกลุ่ม + จับค่า

ใช้ `( )` เพื่อ
1. **จัดกลุ่ม** — เอา quantifier ครอบหลายตัวอักษร เช่น `(ab)+` = "ab" ซ้ำๆ
2. **Capture** — ดึงค่าออกมาใช้ภายหลัง

### Alternation `|` — "หรือ"

| Pattern | Match |
|---------|-------|
| `cat\|dog` | "cat" หรือ "dog" |
| `(jpg\|png\|gif)` | นามสกุลรูปภาพ |
| `^(GET\|POST\|PUT)` | บรรทัดขึ้นต้นด้วย HTTP method เหล่านี้ |

### Non-capturing group `(?: ... )`

ถ้าต้องการแค่จัดกลุ่ม โดยไม่อยาก "capture" ค่า ใช้ `(?:...)` แทน — เร็วกว่า + ไม่ทำให้ index ของ group เลื่อน

### Named group `(?P<name>...)` (Python) หรือ `(?<name>...)` (อื่นๆ)

ตั้งชื่อ group ให้อ้างถึงได้ง่ายขึ้น ใน Splunk จะใช้รูปแบบ `(?<name>...)`

### Backreference `\1`, `\2`, ...

อ้างถึงค่า group ที่ capture ไว้ก่อนหน้า เช่น `(\w+) \1` = จับคำที่ซ้ำกันสองครั้ง

In [ ]:
import re

# Group + quantifier
print(re.findall(r"(ab)+", "ababab xx ab"))   # ['ab', 'ab']  (เก็บแค่ group สุดท้ายของแต่ละ match)

# Alternation
print(re.findall(r"cat|dog|bird", "I have a cat, a dog, and a fish"))
# ['cat', 'dog']

# Capture — ดึงปี/เดือน/วันออกจากวันที่
date_str = "2026-05-23"
m = re.match(r"(\d{4})-(\d{2})-(\d{2})", date_str)
print(m.group(0))    # '2026-05-23'  (ทั้ง match)
print(m.group(1))    # '2026'         (group 1)
print(m.group(2))    # '05'
print(m.group(3))    # '23'
print(m.groups())    # ('2026', '05', '23')

# Named groups — อ่านง่ายกว่ามาก
m = re.match(r"(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})", date_str)
print(m.group("year"), m.group("month"), m.group("day"))
print(m.groupdict())   # {'year': '2026', 'month': '05', 'day': '23'}

# Backreference — หาคำที่พิมพ์ซ้ำ (typo คลาสสิก)
text = "this is is a test test of duplicate words"
print(re.findall(r"\b(\w+)\s+\1\b", text))   # ['is', 'test']

## 7. Lookaround — Lookahead & Lookbehind

Lookaround คือ "เงื่อนไขประกอบ" ที่เช็คว่ารอบๆ มีอะไร **แต่ไม่นับเข้าไปใน match**

| Syntax | ชื่อ | ความหมาย |
|--------|------|---------|
| `X(?=Y)`  | positive lookahead   | X ที่ "ตามด้วย" Y |
| `X(?!Y)`  | negative lookahead   | X ที่ "ไม่ตามด้วย" Y |
| `(?<=Y)X` | positive lookbehind  | X ที่ "นำหน้าด้วย" Y |
| `(?<!Y)X` | negative lookbehind  | X ที่ "ไม่นำหน้าด้วย" Y |

### ทำไมต้องใช้?

สมมติอยากดึงตัวเลขออกจาก "Price: $1,299" โดยไม่เอา `$` มาด้วย → ใช้ lookbehind

```
(?<=\$)\d+   →  match "1" จาก "$1,299" (ดึงตัวเลขหลัง $)
```

หรืออยากดึง username จาก email โดยไม่เอา `@domain.com` มาด้วย → ใช้ lookahead

```
\w+(?=@)   →  match "john" จาก "john@example.com"
```

In [ ]:
import re

# Lookahead — username จาก email
emails = "Contact john@example.com or alice@gmail.com"
print(re.findall(r"\w+(?=@)", emails))   # ['john', 'alice']

# Lookbehind — ดึงราคา (ไม่เอา $ มาด้วย)
print(re.findall(r"(?<=\$)\d+", "Items: $10, $250, $1000"))   # ['10', '250', '1000']

# Negative lookahead — หาคำ "log" ที่ "ไม่ใช่" login/logout
print(re.findall(r"\blog(?!in|out)\w*", "log login logout logger"))
# ['log', 'logger']

# Negative lookbehind — หา word "Server" ที่ไม่ได้นำหน้าด้วย "Web"
print(re.findall(r"(?<!Web)Server", "WebServer ApiServer Server MyServer"))
# ['Server', 'Server', 'Server']  ← จาก "ApiServer", "Server", "MyServer"

## 8. ตัวอย่างจริง — Splunk + Security

### 8.1 Patterns ที่ใช้บ่อย

| สิ่งที่อยากจับ | Regex |
|--------------|-------|
| IPv4 address       | `\b(?:\d{1,3}\.){3}\d{1,3}\b` |
| Email              | `[\w.+-]+@[\w-]+\.[\w.-]+` |
| URL (พื้นฐาน)      | `https?://[^\s]+` |
| MAC address        | `(?:[0-9A-Fa-f]{2}[:-]){5}[0-9A-Fa-f]{2}` |
| MD5 hash           | `\b[a-fA-F0-9]{32}\b` |
| SHA-256 hash       | `\b[a-fA-F0-9]{64}\b` |
| Date (YYYY-MM-DD)  | `\d{4}-\d{2}-\d{2}` |
| Time (HH:MM:SS)    | `\d{2}:\d{2}:\d{2}` |
| Windows path       | `[A-Z]:\\(?:[^\\\s]+\\)*[^\\\s]+` |
| Quoted string      | `"[^"]*"` |

### 8.2 ใน Splunk — `rex` command

ใช้ `rex` เพื่อ extract field จาก raw log แบบ runtime

```
... | rex field=_raw "user=(?<username>\w+)"
... | rex "from\s+(?<src_ip>\d+\.\d+\.\d+\.\d+)"
... | rex "(?<method>GET|POST|PUT|DELETE)\s+(?<uri>\S+)"
```

หมายเหตุ:
- Splunk ใช้ PCRE — syntax คล้าย Python มาก
- ใช้ `(?<name>...)` (ไม่มี `P`)
- Field ที่ extract ออกมาใช้ได้ทันทีใน pipeline ต่อไป

In [ ]:
import re

# === Apache/Nginx access log ===
log = '192.168.1.10 - alice [23/May/2026:14:30:55 +0700] "GET /admin/login HTTP/1.1" 401 312'

pattern = (
    r"(?P<ip>\d+\.\d+\.\d+\.\d+)\s+-\s+"
    r"(?P<user>\S+)\s+"
    r"\[(?P<time>[^\]]+)\]\s+"
    r'"(?P<method>\w+)\s+(?P<uri>\S+)\s+(?P<proto>[^"]+)"\s+'
    r"(?P<status>\d+)\s+(?P<size>\d+)"
)
m = re.search(pattern, log)
print(m.groupdict())
# {'ip': '192.168.1.10', 'user': 'alice', 'time': '23/May/2026:14:30:55 +0700',
#  'method': 'GET', 'uri': '/admin/login', 'proto': 'HTTP/1.1', 'status': '401', 'size': '312'}

# === ดึง IP ทั้งหมดจาก log ก้อนใหญ่ ===
big_log = """
Failed login from 10.0.0.5 to host server01
Successful login from 192.168.1.100
Connection from 8.8.8.8 dropped
"""
ips = re.findall(r"\b(?:\d{1,3}\.){3}\d{1,3}\b", big_log)
print(ips)   # ['10.0.0.5', '192.168.1.100', '8.8.8.8']

# === Detect รูปแบบ SQL injection ง่ายๆ ===
suspicious = [
    "SELECT * FROM users WHERE id=1",
    "id=1 OR 1=1--",
    "username=admin' --",
    "normal_input_value",
]
sqli_pattern = re.compile(r"(\bOR\b\s+\d+=\d+|--\s*$|'\s*OR\s*')", re.IGNORECASE)
for s in suspicious:
    if sqli_pattern.search(s):
        print(f"[!] SQLi suspect: {s}")

## 9. แบบฝึกหัด — ลองเขียน Regex เอง

ลองเขียน regex ในเซลล์ถัดไป แล้วรันดูคำตอบ

| # | โจทย์ |
|---|------|
| 1 | match Thai mobile phone เช่น `081-234-5678` หรือ `0812345678` |
| 2 | ดึง username + domain ออกจาก email แยกกัน |
| 3 | match วันที่ในรูปแบบ DD/MM/YYYY (เดือนต้อง 01-12, วันต้อง 01-31 แบบหลวมๆ) |
| 4 | ดึง URL ทั้งหมดออกจากข้อความ |
| 5 | match รหัสผ่านที่ "ปลอดภัย" — อย่างน้อย 8 ตัว, มีตัวพิมพ์ใหญ่, เล็ก, ตัวเลข, อักขระพิเศษ |
| 6 | ดึง key=value ทุกคู่ออกจาก log เช่น `user=alice action=login status=ok` |
| 7 | match private IP (10.x, 192.168.x, 172.16-31.x) |

In [ ]:
import re

# โจทย์ 1: เบอร์มือถือไทย
phones_text = "ติดต่อ 081-234-5678 หรือ 0823456789 หรือ 02-123-4567 (เบอร์บ้าน)"
# TODO: เขียน regex ของคุณตรงนี้
pattern_1 = r"0[689]\d[-\s]?\d{3}[-\s]?\d{4}"
print("1:", re.findall(pattern_1, phones_text))


# โจทย์ 2: แยก username + domain
emails = ["john.doe@example.com", "alice+work@gmail.com", "bob_99@sub.domain.co.th"]
pattern_2 = r"(?P<user>[\w.+-]+)@(?P<domain>[\w.-]+)"
for e in emails:
    m = re.match(pattern_2, e)
    print("2:", m.groupdict())


# โจทย์ 3: วันที่ DD/MM/YYYY
dates = "วันเกิด 23/05/2026 deadline 01/13/2026 invalid 32/01/2026"
pattern_3 = r"\b(0[1-9]|[12]\d|3[01])/(0[1-9]|1[0-2])/\d{4}\b"
print("3:", re.findall(pattern_3, dates))


# โจทย์ 4: ดึง URL
text_4 = "ดูที่ https://example.com/path?q=1 หรือ http://test.org และ ftp://nope.com"
pattern_4 = r"https?://[^\s]+"
print("4:", re.findall(pattern_4, text_4))


# โจทย์ 5: รหัสผ่านปลอดภัย (ใช้ lookahead — เทคนิคยอดนิยม)
passwords = ["Aa1!aaaa", "weakpass", "NOLOWER1!", "NoDigit!", "Aa1!"]
pattern_5 = r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[^\w\s]).{8,}$"
for p in passwords:
    print(f"5: {p:12} → {'OK' if re.match(pattern_5, p) else 'WEAK'}")


# โจทย์ 6: key=value pairs
log = "user=alice action=login status=ok ip=10.0.0.5 ts=2026-05-23"
pattern_6 = r"(\w+)=(\S+)"
print("6:", dict(re.findall(pattern_6, log)))


# โจทย์ 7: Private IP
ips = ["10.0.0.1", "192.168.1.100", "172.16.5.5", "172.32.0.1", "8.8.8.8", "172.20.10.10"]
pattern_7 = r"^(?:10\.|192\.168\.|172\.(?:1[6-9]|2\d|3[01])\.)\d+\.\d+$"
for ip in ips:
    print(f"7: {ip:18} → {'PRIVATE' if re.match(pattern_7, ip) else 'public'}")

## สรุป Cheatsheet

```
ตัวอักษรพิเศษ      .  \d \D \w \W \s \S \b
Character class    [abc]  [a-z]  [^abc]
Quantifiers        *  +  ?  {n}  {n,}  {n,m}    (เติม ? = lazy)
Anchors            ^  $  \b
Groups             ( )  (?: )  (?P<name> )  \1
Alternation        |
Lookahead          (?= )  (?! )
Lookbehind         (?<= ) (?<! )
Escape             \. \? \\ \( \) ...
```

### Python `re` ที่ใช้บ่อย

```python
re.match(pattern, text)        # match จากต้น string
re.search(pattern, text)       # หา match แรกที่ไหนก็ได้
re.findall(pattern, text)      # หา match ทั้งหมด → list
re.finditer(pattern, text)     # เหมือน findall แต่คืน iterator ของ Match
re.sub(pattern, repl, text)    # แทนที่
re.split(pattern, text)        # split ด้วย pattern

# Flags
re.IGNORECASE  # หรือ re.I — ไม่แคร์ตัวพิมพ์
re.MULTILINE   # หรือ re.M — ^$ ทำงานต่อบรรทัด
re.DOTALL      # หรือ re.S — . match newline ด้วย
re.VERBOSE     # หรือ re.X — เขียน regex หลายบรรทัด+คอมเมนต์ได้
```

### Tips สุดท้าย

1. **ใช้ raw string `r"..."`** เสมอใน Python — ไม่งั้นต้อง escape `\` ซ้อน
2. **ฝึกกับ regex101.com** — มี explanation ทุก token
3. **อย่า over-engineer** — บางทีใช้ `str.split()` หรือ `in` ก็พอ
4. **Regex ไม่เหมาะกับ HTML/JSON ที่ซับซ้อน** — ใช้ parser จริงๆ ดีกว่า
5. **Performance:** ระวัง catastrophic backtracking (`(a+)+`) — pattern แย่ๆ ทำให้ engine ค้างได้

Happy regexing! 🎯